# Monday: comparing replies with some embeddings

Which replies talk about similar things? In Week 8 each reply became a row of TF-IDF weights. This week each reply also becomes an embedding: a row of numbers made by a trained language model. You compare the two methods on the same AITA replies.

Work with a partner. Write code in the cells that start with a comment, and use the check comments to confirm your result. Parts 1–4 are the core. Switch who types after Part 2. Make a copy named `week09_live_workspace_mine.ipynb`.

In [ ]:
# Install sentence-transformers and the other optional NLP packages into the course environment.
# Run this once on your own computer. It can take a few minutes. Skip it on Colab, which already has them.
!uv sync --frozen --group optional-nlp

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## 1. Sentences as embeddings

In [ ]:
sentences = [
    "The cat sat on the mat.",
    "The cat sat on the rug.",
    "The capital of France is Paris.",
    "A kitten rested on the carpet.",
]

In [ ]:
# Load a trained embedding model. The first run downloads about 90 MB.
model = SentenceTransformer("all-MiniLM-L6-v2")

`model.encode()` takes a list of texts and returns one row of numbers for each text. Encode `sentences` as `embeddings` and show its shape.

In [ ]:
# Encode the sentences and show the shape. Check: (4, 384)

In [ ]:
embeddings[0][:8]  # the first 8 of the 384 numbers for sentence 0

In a TF-IDF table every column is a word. What is column 0 here?

Use `cosine_similarity` to score every pair of embeddings, as with TF-IDF rows in Week 8. Put the result in a DataFrame and round to 2 decimals.

In [ ]:
# Score every pair of embeddings. Check: sentences 0 and 3 score 0.61

In [ ]:
# The same comparison with TF-IDF rows
tfidf = TfidfVectorizer(stop_words="english")
tfidf_rows = tfidf.fit_transform(sentences)
pd.DataFrame(cosine_similarity(tfidf_rows)).round(2)  # Check: sentences 0 and 3 score 0.0

Sentences 0 and 3 share no words once stop words are removed. Which method still scores them as similar? Which sentence pair do both methods score near 0?

### Test the model

In [ ]:
pair = ["You are not wrong.", "You are wrong."]
cosine_similarity(model.encode(pair)).round(2)  # Check: 0.83

The two sentences give opposite verdicts. Change the second sentence to `"You did nothing wrong."` and run the cell again. Then try a pair of your own. What does a high score tell you about two sentences, and what does it not tell you?

## 2. Embed the AITA replies

These are the Week 8 replies. `reset_index(drop=True)` numbers the rows 0 to 1034, so a row number in `replies` matches a row in the score tables below.

In [ ]:
url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
comments = pd.read_csv(url)
pd.set_option("display.max_colwidth", 300)  # show more of each reply

comments["marker"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")
replies = comments.dropna(subset=["marker"]).reset_index(drop=True)
replies["marker"].value_counts()  # Check: NTA 657, YTA 378

Encode every reply as `reply_embeddings`. `model.encode()` needs a list, so pass `replies["body"].tolist()`. This can take a minute.

In [ ]:
# Make reply_embeddings and show its shape. Check: (1035, 384)

In [ ]:
# TF-IDF rows for the same replies, with the Week 8 settings
reply_tfidf = TfidfVectorizer(stop_words="english", min_df=5).fit_transform(replies["body"])
reply_tfidf.shape  # Check: (1035, 773)

Score every pair of replies with each method. Call the results `emb_scores` and `tfidf_scores`.

In [ ]:
# Make emb_scores and tfidf_scores. Check: both have shape (1035, 1035)

## 3. Find the nearest replies

Row 92 of `emb_scores` holds the scores between reply 92 and every reply.

In [ ]:
anchor = 92
print(replies.loc[anchor, "idstr"])  # Check: t1_ej0kc3p
print(replies.loc[anchor, "body"])

`argsort()` lists row numbers from the lowest score to the highest, and `[::-1]` reverses that order. The first entry is then the anchor itself (score 1), so `[1:4]` skips it and keeps the next three.

In [ ]:
nearest = emb_scores[anchor].argsort()[::-1][1:4]
print(emb_scores[anchor, nearest].round(2))  # Check: [0.58 0.54 0.52]
replies.loc[nearest, ["idstr", "marker", "body"]]

Find the three nearest replies with `tfidf_scores`.

In [ ]:
# Find the three nearest replies with TF-IDF. Check: the first is t1_elgvbqm

Read both lists. Which list has replies about a situation like the anchor's? Which words does the anchor share with t1_elgvbqm? Then set `anchor = 219` and run Part 3 again.

## 4. Search with your own sentence

Encode a query with the same model and compare it with every reply. With two arguments, `cosine_similarity(a, b)` scores each row of `a` against each row of `b`. The result has one row, and `[0]` takes it. There is no anchor to skip, so keep the first five.

In [ ]:
query = "He expects his wife to do all the housework."
query_embedding = model.encode([query])

In [ ]:
# Score the query against reply_embeddings and show the five nearest replies. Check: the first is t1_elhl65l

In [ ]:
# The Week 8 word search for the same topic
replies["body"].str.contains(r"\bhousework\b", case=False).sum()  # Check: 3

How many of the five replies use the word *housework*? Now search for `"My roommate keeps eating my food."` Are the results about roommates? What does the search return when no reply is about your topic?

## If you have time

### A. Do nearest replies share the verdict?

For each reply, find its nearest other reply and check whether the two have the same marker. `fill_diagonal` sets each reply's score with itself to 0, and `argmax(axis=1)` gives the highest-scoring row for each reply.

In [ ]:
import numpy as np

scores = emb_scores.copy()  # a copy, so emb_scores keeps its 1s
np.fill_diagonal(scores, 0)
nearest_all = scores.argmax(axis=1)
same = replies["marker"].values == replies["marker"].values[nearest_all]
same.mean()  # Check: 0.79

Repeat with `tfidf_scores`. If replies were paired at random, about 0.54 of pairs would share a marker.

In [ ]:
# Repeat for tfidf_scores. Check: 0.65

### B. Remove the verdict

Both methods may be matching on the leading NTA or YTA. Remove it, encode again, and repeat A.

In [ ]:
no_marker = replies["body"].str.replace(r"^(NTA|YTA)\b", "", regex=True, case=False)
no_marker.head(3)

In [ ]:
# Encode no_marker, score every pair, and repeat A. Check: 0.57

How much did the share drop? What were the embeddings matching on before?

### C. Build a meaning axis

Average the embeddings of two groups of words and subtract one average from the other. The result is a direction, or axis, from one group to the other. `cosine_similarity` with that axis scores where any text falls along it.

In [ ]:
female_words = ["she", "her", "woman", "girl", "wife", "mother", "daughter", "sister"]
male_words = ["he", "him", "man", "boy", "husband", "father", "son", "brother"]

# The axis points from the average male word to the average female word
gender_axis = model.encode(female_words).mean(axis=0) - model.encode(male_words).mean(axis=0)

In [ ]:
roles = ["king", "queen", "nurse", "engineer", "victim", "aggressor", "supporter", "critic"]
role_scores = cosine_similarity(model.encode(roles), [gender_axis])[:, 0]
pd.Series(role_scores, index=roles).round(2).sort_values()  # Check: king -0.08, queen 0.25

Build `moral_axis` the same way, from `justified_words` to `unjustified_words`, and score the actions. Positive means closer to the justified words.

In [ ]:
justified_words = ["justified", "reasonable", "right", "fair", "understandable"]
unjustified_words = ["unjustified", "unreasonable", "wrong", "unfair", "selfish"]
actions = ["refused", "demanded", "asked", "yelled", "ignored", "helped", "apologized", "explained"]

In [ ]:
# Make moral_axis and score the actions. Check: refused -0.13, helped 0.09

Which scores match your sense of the word? Which one would you not have expected?

In [ ]:
# Score every reply, without its NTA or YTA, on the moral axis
replies["moral"] = cosine_similarity(model.encode(no_marker.tolist()), [moral_axis])[:, 0]
replies.groupby("marker")["moral"].mean().round(3)  # Check: NTA -0.048, YTA -0.049

NTA replies say the poster was justified and YTA replies say they were not. Does the axis separate the two groups? Why might an axis built from ten words not measure the judgment in a whole reply?

Save the notebook, restart the kernel, and run every cell.